# 03 — Évaluation du modèle

Ce notebook charge un checkpoint entraîné et calcule les métriques sur le split `validation`.

Métriques : **BLEU**, **ROUGE**

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("seq2seq-attention"):
        !git clone https://github.com/Arsenicophe/seq2seq-attention.git
    else:
        !git -C seq2seq-attention pull
    os.chdir("seq2seq-attention/notebooks")
    !pip install -q sacrebleu rouge-score spacy
    !python -m spacy download en_core_web_sm -q
    !python -m spacy download fr_core_news_sm -q

src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

## Imports

In [ ]:
import sys, yaml
sys.path.append("../src")

import torch
import spacy
from datasets import load_dataset
from tqdm import tqdm

from encoder  import Encoder
from decoder  import Decoder
from seq2seq  import Seq2Seq
from data     import Vocab
from metrics.bleu  import corpus_bleu
from metrics.rouge import corpus_rouge
from sampling.beam_search import beam_search_decode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

## Chargement du checkpoint

In [ ]:
CHECKPOINT_PATH = "../checkpoints/checkpoint_epoch_10.pt"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
print(f"Checkpoint chargé — epoch {checkpoint['epoch']}, loss {checkpoint['loss']:.4f}")

In [ ]:
# On reconstruit le même vocab qu'à l'entraînement (mêmes données, même min_freq)
nlp_en = spacy.load("en_core_web_sm", disable=["ner", "parser"])
nlp_fr = spacy.load("fr_core_news_sm", disable=["ner", "parser"])

def batch_tokenize(nlp, sentences, batch_size=512):
    return [
        [tok.text.lower() for tok in doc]
        for doc in tqdm(nlp.pipe(sentences, batch_size=batch_size), total=len(sentences))
    ]

N    = cfg["data"]["n_train"]
ds   = load_dataset("Helsinki-NLP/opus-100", "en-fr")
train = ds["train"]

src_train = [ex["translation"]["en"] for ex in train.select(range(N))]
trg_train = [ex["translation"]["fr"] for ex in train.select(range(N))]

src_vocab = Vocab.build(batch_tokenize(nlp_en, src_train), min_freq=cfg["data"]["min_freq"])
trg_vocab = Vocab.build(batch_tokenize(nlp_fr, trg_train), min_freq=cfg["data"]["min_freq"])

# Reconstruction du modèle + chargement des poids
encoder = Encoder(vocab_size=len(src_vocab), **cfg["encoder"])
decoder = Decoder(vocab_size=len(trg_vocab), **cfg["decoder"])
model   = Seq2Seq(encoder, decoder, device).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Modèle prêt — Vocab EN : {len(src_vocab):,} | Vocab FR : {len(trg_vocab):,}")

## Préparation des données de validation

In [ ]:
val = ds["validation"]

src_sentences = [ex["translation"]["en"] for ex in val]
trg_sentences = [ex["translation"]["fr"] for ex in val]

print(f"Phrases de validation : {len(src_sentences):,}")

In [ ]:
src_tokens = batch_tokenize(nlp_en, src_sentences)
trg_tokens = batch_tokenize(nlp_fr, trg_sentences)

## Génération des traductions

In [ ]:
EOS_IDX = 2  # convention data.py

predictions = []
references  = []

model.eval()
with torch.no_grad():
    for src_tok, trg_tok in tqdm(zip(src_tokens, trg_tokens), total=len(src_tokens)):
        # Source → tensor (src_len, 1)
        src_ids    = src_vocab.encode(src_tok) + [EOS_IDX]
        src_tensor = torch.tensor(src_ids, device=device).unsqueeze(1)
        src_length = torch.tensor([len(src_ids)])

        # Beam search → liste d'indices
        pred_ids = beam_search_decode(
            model, src_tensor, src_length, beam_size=5, device=device
        )

        # Indices → tokens (on exclut les tokens spéciaux)
        pred_tokens = [trg_vocab.itos[i] for i in pred_ids
                       if i not in (0, 1, 2, 3)]  # PAD SOS EOS UNK

        predictions.append(pred_tokens)
        references.append([trg_tok])  # corpus_bleu attend une liste de références

print(f"Traductions générées : {len(predictions)}")

## Calcul des métriques

In [ ]:
bleu = corpus_bleu(predictions, references)
print(f"BLEU : {bleu:.2f}")

In [ ]:
# corpus_rouge attend des listes de tokens (pas de liste de listes pour references ici)
refs_flat = [r[0] for r in references]
rouge = corpus_rouge(predictions, refs_flat, n=1)
print(f"ROUGE-1  precision={rouge.precision:.3f}  recall={rouge.recall:.3f}  f1={rouge.f1:.3f}")